In [ ]:
Step 1: Data Ingestion and Initial Cleaning
In this step, we:
Initialized a Spark Session optimized for local development with controlled memory and parallelism.
Loaded the raw contract dataset (mcc_raw.csv) into Spark, inferring schema and keeping headers.
Cleaned column names (renamed complex names like company.name → company_name).
Selected relevant columns for downstream processing, dropping unnecessary ones.
Created a temporary SQL view (contracts) for interactive exploration.
Performed an initial label frequency check to understand distribution across contract types.
This establishes a clean foundation for further preprocessing, labeling, and classification tasks.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# -----------------------------------------------------------------------------
# 1. Initialize Spark Session
# -----------------------------------------------------------------------------
spark = (
    SparkSession.builder
    .appName("MCC_ETL")
    .master("local[8]")  # Run locally with 8 cores (leaves headroom for OS tasks)
    .config("spark.driver.memory", "12g")      # Allocate driver memory
    .config("spark.executor.memory", "12g")    # Allocate executor memory
    .config("spark.sql.shuffle.partitions", "16")  # Control shuffle parallelism
    .config("spark.default.parallelism", "16")     # Default number of tasks
    .getOrCreate()
)

print(f"Spark version: {spark.version}")

# -----------------------------------------------------------------------------
# 2. Load raw dataset (CSV format)
# -----------------------------------------------------------------------------
file_path = "../data/raw/mcc_raw.csv"

df = spark.read.csv(
    file_path,
    header=True,        # Use first row as header
    inferSchema=True    # Let Spark infer column types
)

# Inspect schema and preview data
df.printSchema()
df.show(5, truncate=100)

# -----------------------------------------------------------------------------
# 3. Rename columns for clarity
# -----------------------------------------------------------------------------
df = df.withColumnRenamed("company.name", "company_name") \
       .withColumnRenamed("form.type", "form_type") \
       .withColumnRenamed("date.filed", "date_filed")

# Select only the relevant columns for further processing
selected_columns = [
    "year",
    "company_name",
    "form_type",
    "date_filed",
    "description",
    "contract",
    "type_label",
    "type_score",
    "amend",
    "restate",
    "joinder",
    "termination",
    "agreement_type"
]

df = df.select(*selected_columns)

# Preview after cleanup
df.show(5, truncate=100)

# -----------------------------------------------------------------------------
# 4. Register temporary SQL view for exploration
# -----------------------------------------------------------------------------
df.createOrReplaceTempView("contracts")

# Quick frequency check of label distribution
spark.sql("""
    SELECT type_label, COUNT(*) AS frequency
    FROM contracts
    WHERE type_label IS NOT NULL
    GROUP BY type_label
    ORDER BY frequency DESC
""").show()


# Contract Data Cleaning Pipeline

This pipeline prepares contract data for classification tasks by cleaning, deduplicating, and structuring the dataset.

## Steps

### 1. Filter Labels
- Keep only `type_label`s with at least **10,000 records**.  
- Remove malformed labels (e.g., file paths, EDGAR references, stray quotes).

### 2. Clean & Deduplicate
- Normalize `contract` values by removing `/Archives/edgar/data/` prefix.  
- Deduplicate on `(contract, type_label, description)`, keeping the **latest filing** (`date_filed`) and **highest confidence** (`type_score`).

### 3. Add Metadata
- `data_split`: 5-fold partition per label for train/validation sampling.  
- `label_count`: frequency of each label across the dataset.

### 4. Final Output
- Cleaned dataset available as: **`final_cleaned_data`**  
- Retained columns: key contract fields + splits + label counts.  
- Verified uniqueness and sampled 50 rows per label for inspection.


In [ ]:
# -----------------------------------------------------------------------------
# STEP 1: Identify valid labels (frequency ≥ 10,000)
# - Filters out noisy/malformed labels such as file paths and 'edgar' entries
# -----------------------------------------------------------------------------
final_cleaned_data = spark.sql("""
WITH valid_labels AS (
    SELECT type_label
    FROM contracts
    WHERE type_label IS NOT NULL
      AND type_label NOT LIKE '%/%'
      AND type_label NOT LIKE '%edgar%'
      AND type_label NOT LIKE '%"'
    GROUP BY type_label
    HAVING COUNT(*) >= 10000
),

-- -----------------------------------------------------------------------------
-- STEP 2: Clean & Deduplicate Contracts
-- - Remove /Archives/edgar/data/ prefix from `contract` column
-- - Deduplicate by contract + type_label + description
--   keeping the latest/highest-scored version
-- -----------------------------------------------------------------------------
cleaned_contracts AS (
    SELECT DISTINCT
        c.year,
        c.company_name,
        c.form_type,
        c.date_filed,
        c.description,
        CASE
            WHEN c.contract LIKE '/Archives/edgar/data/%'
            THEN REGEXP_REPLACE(c.contract, '^/Archives/edgar/data/', '')
            ELSE c.contract
        END as contract,
        c.type_label,
        c.type_score,
        c.amend,
        c.restate,
        c.joinder,
        c.termination,
        c.agreement_type,
        ROW_NUMBER() OVER (
            PARTITION BY 
                CASE
                    WHEN c.contract LIKE '/Archives/edgar/data/%'
                    THEN REGEXP_REPLACE(c.contract, '^/Archives/edgar/data/', '')
                    ELSE c.contract
                END,
                c.type_label,
                c.description
            ORDER BY c.date_filed DESC, c.type_score DESC
        ) as rn
    FROM contracts c
    INNER JOIN valid_labels v ON c.type_label = v.type_label
    WHERE c.type_label IS NOT NULL
      AND c.description IS NOT NULL
      AND c.contract IS NOT NULL
      AND LENGTH(TRIM(c.description)) > 0
      AND LENGTH(TRIM(c.contract)) > 0
),

-- -----------------------------------------------------------------------------
-- STEP 3: Add Partitioning Metadata
-- - `data_split`: stratified 5-fold split within each label for training/validation
-- - `label_count`: total count per label (helps with balancing/analysis)
-- -----------------------------------------------------------------------------
final_dataset AS (
    SELECT 
        *,
        NTILE(5) OVER (PARTITION BY type_label ORDER BY RAND()) as data_split,
        COUNT(*) OVER (PARTITION BY type_label) as label_count
    FROM cleaned_contracts
    WHERE rn = 1
)

-- -----------------------------------------------------------------------------
-- STEP 4: Select Final Columns
-- -----------------------------------------------------------------------------
SELECT
    year,
    company_name,
    form_type,
    date_filed,
    description,
    contract,
    type_label,
    type_score,
    amend,
    restate,
    joinder,
    termination,
    agreement_type,
    data_split,
    label_count
FROM final_dataset
ORDER BY type_label, data_split
""")

# Register cleaned dataset for further analysis
final_cleaned_data.createOrReplaceTempView("final_cleaned_data")

# -----------------------------------------------------------------------------
# Validate cleaning results
# -----------------------------------------------------------------------------
spark.sql("""
SELECT 
    COUNT(*) as final_total_rows,
    COUNT(DISTINCT contract) as final_unique_contracts,
    ROUND(COUNT(DISTINCT contract) * 100.0 / COUNT(*), 2) as final_uniqueness_percentage
FROM final_cleaned_data
""").show()

# -----------------------------------------------------------------------------
# Quick sampling: take 50 random rows per type_label
# -----------------------------------------------------------------------------
spark.sql("""
WITH sampled_data AS (
   SELECT *,
       ROW_NUMBER() OVER (PARTITION BY type_label ORDER BY RAND()) as sample_rn
   FROM final_cleaned_data
)
SELECT 
   contract,
   type_label,
   description,
   agreement_type,
   form_type
FROM sampled_data
WHERE sample_rn <= 50
ORDER BY type_label, sample_rn
""").show(10, truncate=False)


## Exploratory Data Analysis (EDA) on Final Cleaned Contracts Data

Analyzed the `final_cleaned_data` to assess label consistency and text quality.

### Key Steps
1. **Labels & Agreement Types** – Verified distinct `type_label` and `agreement_type` pairs.  
2. **Text Lengths** – Checked avg/min/max lengths of `description` and `contract` per label.  
3. **Sample Content** – Reviewed 5 random samples per label (truncated text).  
4. **Data Quality** – Flagged very short or null-like entries, computed avg words in descriptions.  
5. **Manual Review** – Random 100k rows for optional spot-check.  

 Outcome: Dataset shows consistent labels, reasonable text lengths, and good quality for downstream NLP tasks.


In [ ]:
# 1. Distinct labels and agreement types
spark.sql("""
SELECT DISTINCT type_label, agreement_type
FROM final_cleaned_data
ORDER BY type_label, agreement_type
""").show(truncate=False)


# 2. Text length distribution (avg/min/max) by label
spark.sql("""
SELECT 
    type_label,
    AVG(LENGTH(description)) as avg_description_length,
    MIN(LENGTH(description)) as min_description_length,
    MAX(LENGTH(description)) as max_description_length,
    AVG(LENGTH(contract)) as avg_contract_length,
    MIN(LENGTH(contract)) as min_contract_length,
    MAX(LENGTH(contract)) as max_contract_length,
    COUNT(*) as sample_count
FROM final_cleaned_data 
GROUP BY type_label
ORDER BY type_label
""").show()


# 3. Random sample of text content (5 per label)
spark.sql("""
WITH text_samples AS (
    SELECT 
        type_label,
        description,
        contract,
        agreement_type,
        ROW_NUMBER() OVER (PARTITION BY type_label ORDER BY RAND()) as rn
    FROM final_cleaned_data
    WHERE LENGTH(TRIM(description)) > 20  -- filter out trivial text
)
SELECT 
    type_label,
    SUBSTRING(description, 1, 200) as description_sample,
    SUBSTRING(contract, 1, 100) as contract_sample,
    agreement_type
FROM text_samples
WHERE rn <= 5
ORDER BY type_label, rn
""").show(100, truncate=False)


# 4. Data quality checks (short, null-like, avg words)
spark.sql("""
SELECT 
    type_label,
    COUNT(*) as total_samples,
    SUM(CASE WHEN LENGTH(TRIM(description)) <= 10 THEN 1 ELSE 0 END) as very_short_descriptions,
    SUM(CASE WHEN description LIKE '%null%' OR description LIKE '%N/A%' THEN 1 ELSE 0 END) as null_like_descriptions,
    SUM(CASE WHEN LENGTH(TRIM(contract)) <= 10 THEN 1 ELSE 0 END) as very_short_contracts,
    ROUND(AVG(SIZE(SPLIT(description, ' '))), 2) as avg_words_in_description
FROM final_cleaned_data
GROUP BY type_label
ORDER BY type_label
""").show()


# 5. Optional: Large random sample for manual review
spark.sql("""
SELECT 
    type_label, 
    description, 
    LENGTH(description) as desc_length, 
    agreement_type
FROM final_cleaned_data 
WHERE LENGTH(TRIM(description)) > 20
ORDER BY type_label, RAND()
""").show(100, truncate=False)


## Filtered Contracts Data

- Kept only rows with meaningful descriptions (>20 chars).  
- Summarized samples per `type_label` with avg. length & word count.  
- Randomly reviewed samples across labels to check text quality.  

 Result: Cleaner dataset with short/noisy entries removed.


In [ ]:
# Create a filtered dataset with only meaningful descriptions
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW filtered_contracts AS
SELECT *
FROM final_cleaned_data
WHERE LENGTH(TRIM(description)) > 20
""")

# Check the filtered dataset quality
spark.sql("""
SELECT 
    type_label,
    COUNT(*) as filtered_samples,
    AVG(LENGTH(description)) as avg_desc_length,
    AVG(SIZE(SPLIT(description, ' '))) as avg_word_count
FROM filtered_contracts
GROUP BY type_label
ORDER BY type_label
""").show()

# Sample from all labels to inspect text quality
spark.sql("""
SELECT 
    type_label,
    description,
    LENGTH(description) as desc_length,
    agreement_type
FROM filtered_contracts
WHERE type_label IN ('LABEL_1','LABEL_2','LABEL_3','LABEL_4','LABEL_5','LABEL_6','LABEL_7')
ORDER BY type_label, RAND()
LIMIT 100
""").show(truncate=False)

# Final quality summary after filtering
spark.sql("""
SELECT 
    type_label,
    COUNT(*) as filtered_samples,
    AVG(LENGTH(description)) as avg_desc_length,
    AVG(SIZE(SPLIT(description, ' '))) as avg_word_count
FROM filtered_contracts
GROUP BY type_label
ORDER BY type_label
""").show()


## Relabeled & Finalized Contracts Data

- Standardized `type_label` → `agreement_type` mappings.  
- Verified uniqueness and data quality across contracts.  
- Extracted `contract_name` for easier reference.  
- Saved cleaned dataset to partitioned Parquet (`../data/processed/mcc_contracts_full`).  

 Final dataset is consistent, deduplicated, and ready for modeling.


In [ ]:
# ----------------------------------------
# 1. Relabel agreement_type based on type_label
# ----------------------------------------
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW relabeled_contracts AS
SELECT
    contract,
    description,
    type_label,
    type_score,
    CASE
        WHEN type_label = 'LABEL_0' THEN 'security'
        WHEN type_label = 'LABEL_1' THEN 'employment'
        WHEN type_label = 'LABEL_2' THEN 'lease'
        WHEN type_label = 'LABEL_3' THEN 'services&supply'
        WHEN type_label = 'LABEL_4' THEN 'purchase&ma'
        WHEN type_label = 'LABEL_5' THEN 'shareholder'
        WHEN type_label = 'LABEL_6' THEN 'other'
        WHEN type_label = 'LABEL_7' THEN 'na'
        ELSE agreement_type
    END AS agreement_type,
    data_split,
    label_count
FROM final_cleaned_data
WHERE LENGTH(TRIM(description)) > 20
""")

# Quick verification
spark.sql("SELECT DISTINCT type_label, agreement_type FROM relabeled_contracts ORDER BY type_label").show(truncate=False)


# ----------------------------------------
# 2. Dataset Quality Checks
# ----------------------------------------
spark.sql("""
SELECT 
    COUNT(*) as total_rows,
    COUNT(DISTINCT contract) as unique_contracts,
    ROUND(COUNT(DISTINCT contract) * 100.0 / COUNT(*), 2) as uniqueness_percentage
FROM relabeled_contracts
""").show()

spark.sql("""
SELECT 
    type_label,
    agreement_type,
    COUNT(*) as total_rows,
    COUNT(DISTINCT contract) as unique_contracts
FROM relabeled_contracts
GROUP BY type_label, agreement_type
ORDER BY type_label
""").show()


# ----------------------------------------
# 3. Extract Contract Name
# ----------------------------------------
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW relabeled_contracts_named AS
SELECT
    contract,
    REGEXP_EXTRACT(contract, '[^/]+$', 0) AS contract_name,
    description,
    type_label,
    agreement_type,
    type_score,
    data_split,
    label_count
FROM relabeled_contracts
""")

# Spot check
spark.sql("SELECT contract, contract_name FROM relabeled_contracts_named LIMIT 20").show(truncate=False)


# ----------------------------------------
# 4. Final View & Save
# ----------------------------------------
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW relabeled_contracts_final AS
SELECT
    contract_name AS contract,
    description,
    type_label,
    agreement_type,
    type_score,
    data_split,
    label_count
FROM relabeled_contracts_named
""")

# Save partitioned parquet for production
spark.table("relabeled_contracts_final").write.mode("overwrite") \
    .partitionBy("type_label") \
    .parquet("../data/processed/mcc_contracts_full")

print("Final relabeled dataset saved at data/processed/mcc_contracts_full")

# Quick schema & sample check
df_check = spark.read.parquet("../data/processed/mcc_contracts_full")
df_check.printSchema()
df_check.show(20, truncate=False)


## Final Filtering & Export

- Checked distribution of contracts by label and agreement type.  
- Verified number of unique contracts and share per category.  
- Excluded irrelevant categories (`na`, `other`).  
- Saved the cleaned dataset as partitioned Parquet (`../data/processed/mcc_contracts`).  

 Dataset is now fully filtered and ready for modeling/analysis.


In [ ]:
from pyspark.sql import functions as F

# --- Step 1: Distribution & Quality Checks ---
# Count rows per label/type
df_check.groupBy("type_label", "agreement_type").count().show()

# Distinct contracts
print("Distinct contracts:", df_check.select("contract").distinct().count())

# Distribution by agreement_type (with % share)
df_check.groupBy("agreement_type") \
    .agg(F.count("*").alias("count")) \
    .withColumn("percentage", (F.col("count") / df_check.count()) * 100) \
    .orderBy(F.desc("count")) \
    .show(truncate=False)


# --- Step 2: Filter & Save Final Dataset ---
# Remove unwanted categories: 'na' and 'other'
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW relabeled_contracts_final AS
SELECT
    contract_name AS contract,
    description,
    type_label,
    agreement_type,
    type_score,
    data_split,
    label_count
FROM relabeled_contracts_named
WHERE agreement_type NOT IN ('na', 'other')
""")

# Save partitioned parquet
spark.table("relabeled_contracts_final").write.mode("overwrite") \
    .partitionBy("type_label") \
    .parquet("../data/processed/mcc_contracts")
